### Install dependencies

In [ ]:
pip install ai-core-sdk sap-ai-sdk-gen pandas tabulate

### Load credentials and create the AI Core client

In [ ]:
import time
import json
import os
from IPython.display import clear_output
from ai_core_sdk.ai_core_v2_client import AICoreV2Client
 
# Inline credentials
with open('creds.json') as f:
    credCF = json.load(f)

# Set environment variables
def set_environment_vars(credCF):
    env_vars = {
        'AICORE_AUTH_URL': credCF['url'] + '/oauth/token',
        'AICORE_CLIENT_ID': credCF['clientid'],
        'AICORE_CLIENT_SECRET': credCF['clientsecret'],
        'AICORE_BASE_URL': credCF["serviceurls"]["AI_API_URL"] + "/v2",
        'AICORE_RESOURCE_GROUP': "default" 
    }

    for key, value in env_vars.items():
        os.environ[key] = value
        print(value)

# Create AI Core client instance
def create_ai_core_client(credCF):
    set_environment_vars(credCF)  # Ensure environment variables are set
    return AICoreV2Client(
        base_url=os.environ['AICORE_BASE_URL'],
        auth_url=os.environ['AICORE_AUTH_URL'],
        client_id=os.environ['AICORE_CLIENT_ID'],
        client_secret=os.environ['AICORE_CLIENT_SECRET'],
        resource_group=os.environ['AICORE_RESOURCE_GROUP']
    )

ai_core_client = create_ai_core_client(credCF)

### Create a configuration for RPT-1

In [3]:
from ai_api_client_sdk.models.parameter_binding import ParameterBinding

# Define configuration parameters
scenario_id    = 'foundation-models'
executable_id  = 'aicore-sap'
config_name    = 'sap-rpt-1-small-config'  

config = ai_core_client.configuration.create(
    scenario_id   = scenario_id,
    executable_id = executable_id,
    name          = config_name,
    parameter_bindings = [
        ParameterBinding(key='modelName', value='sap-rpt-1-small')
    ]
)

print(f'Configuration created — ID: {config.id}  Name: {config_name}')

Configuration created — ID: b4d0634e-abc4-47fa-a093-79680beb4db8  Name: sap-rpt-1-small-config


### Create the deployment and wait until it is Running

In [4]:
from ai_api_client_sdk.models.status import Status

deployment = ai_core_client.deployment.create(configuration_id=config.id)
print(f'Deployment created — ID: {deployment.id}')

# Poll until the deployment status is Running
def spinner(check_callback, timeout=300, check_every_n_seconds=10):
    start = time.time()
    while time.time() - start < timeout:
        return_value = check_callback()
        if return_value:
            return return_value
        for char in '|/-\\':
            clear_output(wait=True)
            print(f'Waiting for deployment to become ready... {char}')
            time.sleep(0.2)

def check_ready():
    updated = ai_core_client.deployment.get(deployment.id)
    return updated if updated.status == Status.RUNNING else None

ready_deployment = spinner(check_ready)
print(f'Deployment is ready — Status: {ready_deployment.status}')
print(f'Deployment ID: {ready_deployment.id}')

Waiting for deployment to become ready... \
Deployment is ready — Status: Status.RUNNING
Deployment ID: d5093f6a961b7707


### Load and prepare the dataset

In [5]:
import pandas as pd

# Load your local CSV file — no joins or merging required
df = pd.read_excel('payment-delay-data.xlsx', dtype=str).fillna('')

# Configuration
INDEX_COLUMN  = 'Transaction ID'
TARGET_REG    = 'Days Late'    # regression target — numeric prediction
TARGET_CLS    = 'Risk Score'   # classification target — category + confidence
PREDICT_TOKEN = '[PREDICT]'
RPT_MODEL     = 'sap-rpt-1-small'  # or 'sap-rpt-1-large' for highest accuracy

print(f'Dataset shape : {df.shape}')
print(f'Columns       : {list(df.columns)}')
df.head(3)

Dataset shape : (500, 22)
Columns       : ['Transaction ID', 'Customer ID', 'Days Late', 'Customer Type', 'Industry', 'Region', 'Currency', 'Invoice Amount', 'Due Days', 'Credit Limit', 'Outstanding', 'Credit Usage', 'Relationship Years', 'Previous Delays', 'Avg Days Late', 'Payment Method', 'Economic Indicator', 'Quarter', 'Contact Attempts', 'Has Dispute', 'Dispute Amount', 'Risk Score']


,Transaction ID,Customer ID,Days Late,Customer Type,Industry,Region,Currency,Invoice Amount,Due Days,Credit Limit,...,Relationship Years,Previous Delays,Avg Days Late,Payment Method,Economic Indicator,Quarter,Contact Attempts,Has Dispute,Dispute Amount,Risk Score
0,TXN AZT67X0T,CUST 1014,[PREDICT],Small Business,Education,Middle East,USD,30214.02,15,118859.72,...,5,5,22.7,Credit Card,1.1,Q4,1,False,0,High
1,TXN SY4HFYMZ,CUST 1023,14,Individual,Healthcare,Middle East,USD,1045.84,30,10954.03,...,4,0,3.4,Credit Card,1.09,Q2,2,False,0,High
2,TXN RWO7A9Z2,CUST 1006,10,Small Business,Education,Europe,GBP,17920.85,15,148152.69,...,3,5,22.2,Credit Card,0.9,Q1,2,False,0,[PREDICT]


### Separate context rows from prediction rows

In [6]:
# Context rows — rows where both targets have known (non-PREDICT) values
context_mask = (
    (df[TARGET_REG] != PREDICT_TOKEN) & (df[TARGET_REG] != '') &
    (df[TARGET_CLS] != PREDICT_TOKEN) & (df[TARGET_CLS] != '')
)
context_df = df[context_mask].copy().reset_index(drop=True)

# Prediction rows — rows where either target contains [PREDICT]
predict_mask = (
    (df[TARGET_REG] == PREDICT_TOKEN) |
    (df[TARGET_CLS] == PREDICT_TOKEN)
)
predict_df = df[predict_mask].copy().reset_index(drop=True)

print(f'Context rows (model learns from these) : {len(context_df)}')
print(f'Prediction rows (model fills these in) : {len(predict_df)}')

Context rows (model learns from these) : 469
Prediction rows (model fills these in) : 31


### Build and send the RPT-1 prediction request

In [ ]:
from gen_ai_hub.proxy.native.sap.models import RPTRequest, PredictionConfig, TargetColumn
from gen_ai_hub.proxy.native.sap.client import RPTClient

# Context rows must come first, followed by prediction rows
all_rows = context_df.to_dict(orient='records') + predict_df.to_dict(orient='records')

# Build the prediction request
body = RPTRequest(
    prediction_config=PredictionConfig(
        target_columns=[
            TargetColumn(
                name=TARGET_REG,
                task_type='regression',
                prediction_placeholder=PREDICT_TOKEN
            ),
            TargetColumn(
                name=TARGET_CLS,
                task_type='classification',
                prediction_placeholder=PREDICT_TOKEN
            )
        ]
    ),
    index_column=INDEX_COLUMN,
    rows=all_rows
)
# Call the model
client = RPTClient()
response = client.predict(body=body, model_name=RPT_MODEL)

print(f'Predictions received: {len(response.predictions)}')

Predictions received: 31


### Parse and display the results

In [8]:
from tabulate import tabulate

results = []
for pred in response.predictions:
    results.append({
        INDEX_COLUMN             : pred[INDEX_COLUMN],
        'Predicted Days Late'    : pred[TARGET_REG][0].prediction,
        'Predicted Risk Score'   : pred[TARGET_CLS][0].prediction,
        'Risk Score Confidence'  : round(pred[TARGET_CLS][0].confidence, 4)
    })

print(tabulate(results, headers='keys', tablefmt='grid'))

+------------------+-----------------------+------------------------+-------------------------+
| Transaction ID   |   Predicted Days Late | Predicted Risk Score   |   Risk Score Confidence |
+==================+=======================+========================+=========================+
| TXN AZT67X0T     |              20.2029  | High                   |                    0.72 |
+------------------+-----------------------+------------------------+-------------------------+
| TXN RWO7A9Z2     |              19.9569  | High                   |                    0.69 |
+------------------+-----------------------+------------------------+-------------------------+
| TXN C18O8PW7     |              18.2233  | High                   |                    0.75 |
+------------------+-----------------------+------------------------+-------------------------+
| TXN MQ5R05JP     |              41.6997  | Medium                 |                    0.62 |
+------------------+--------------------

### Export results to CSV

In [9]:
results_df = pd.DataFrame(results)
results_df.to_csv('rpt1_predictions.csv', index=False)
print(f'Results exported to rpt1_predictions.csv ({len(results_df)} rows)')

Results exported to rpt1_predictions.csv (31 rows)
